In [ ]:
import pandas as pd
import unicodedata
import re

# csv als DataFrame einlesen
df = pd.read_csv("project/survey_results_public.csv", index_col="ResponseId")

# bestimmte Spalten entfernen, die weiter nicht gebraucht werden
drop_cols = ['EmploymentAddl', 'LearnCodeChoose', 'LearnCode', 'AILearnHow', 'PurchaseInfluence', 'ToolCountWork', 'ToolCountPersonal', 'LanguageAdmired', 'LanguagesHaveEntry', 'LanguagesWantEntry', 'DatabaseAdmired', 'DatabaseHaveEntry', 'DatabaseWantEntry', 'PlatformAdmired', 'PlatformHaveEntry', 'PlatformWantEntry', 'WebframeAdmired', 'WebframeHaveEntry', 'WebframeWantEntry', 'DevEnvsAdmired', 'DevEnvHaveEntry', 'DevEnvWantEntry', 'OpSysPersonal use', 'OpSysProfessional use', 'OfficeStackAsyncAdmired', 'OfficeStackHaveEntry', 'OfficeStackWantEntry', 'CommPlatformHaveWorkedWith', 'CommPlatformWantToWorkWith', 'CommPlatformAdmired', 'CommPlatformHaveEntr', 'CommPlatformWantEntr', 'AIModelsAdmired', 'AIModelsHaveEntry', 'AIModelsWantEntry', 'AISent', 'AIAcc', 'AIComplex', 'AIToolCurrently partially AI', "AIToolDon't plan to use AI for this task", 'AIToolPlan to partially use AI', 'AIToolPlan to mostly use AI', 'AIToolCurrently mostly AI', 'AIFrustration', 'AIExplain', 'AIAgentChange', 'AgentUsesGeneral', 'AIAgentImpactSomewhat agree', 'AIAgentImpactNeutral', 'AIAgentImpactSomewhat disagree', 'AIAgentImpactStrongly agree', 'AIAgentImpactStrongly disagree', 'AIAgentChallengesNeutral', 'AIAgentChallengesSomewhat disagree', 'AIAgentChallengesStrongly agree', 'AIAgentChallengesSomewhat agree', 'AIAgentChallengesStrongly disagree', 'AIAgentKnowledge', 'AIAgentKnowWrite', 'AIAgentOrchestration', 'AIAgentOrchWrite', 'AIAgentObserveSecure', 'AIAgentObsWrite', 'AIAgentExternal', 'AIAgentExtWrite', 'AIHuman', 'AIOpen']

df = df.drop(columns=drop_cols)
prefixes = ("TechEndorse", "TechOppose", "JobSatPoints", "SO")
df = df.drop(columns=df.columns[df.columns.str.startswith(prefixes)])

# insert Mappings in DatenFrame
def insert_mapped_column(df, base_col, new_col, mapping):
    df.insert(
        df.columns.get_loc(base_col) + 1,
        new_col,
        df[base_col].map(mapping)
    )

# Text vereinheitlichen und Stolperfallen eliminieren
def clean_text(s):
    if pd.isna(s):
        return s
    s = str(s).strip() # Leerzeichen am Anfang und Ende weg
    s = unicodedata.normalize("NFC", s) # Unicode normalisieren
    s = s.replace("–", "-").replace("—", "-") # Bindestriche / Spiegelstriche vereinheitlichen
    s = s.replace("’", "'") # Apostrophen vereinheitlichen
    s = re.sub(r"\s+", " ", s)
    return s

# Zeileninhalt -> lowercase
def to_lowercase(s):
    if pd.isna(s):
        return s
    return s.lower()

# Mulit-Select Spalten cleanen
def clean_multi_select(value):
    if pd.isna(value):
        return []

    splitted = str(value).split(";")

    cleaned = []
    for p in splitted:
        p = clean_text(p)
        if p:
            cleaned.append(p)
    cleaned = list(set(cleaned))
    cleaned.sort()
    return cleaned

#EdLevel splitten um nur EdLevel anzuzeigen
df["EdLevel"] = df["EdLevel"].str.split("(").str[0].str.strip()

#RemoteWork auf numerische Werte mappen
remote_map = {
    "Remote": 0,
    "In-person": 1,
    "Hybrid (some remote, leans heavy to in-person)": 0.75,
    "Hybrid (some in-person, leans heavy to flexibility)": 0.25,
    "Your choice (very flexible, you can come in when you want or just as needed)": 0.5
}
insert_mapped_column(df, "RemoteWork", "RemoteCategoryNum", remote_map)

# Age zu numerischen Werten mappen, immer Mittelwert der ranges
age_map = {
    "Under 18 years old": 17,
    "18-24 years old": 21,
    "25-34 years old": 29,
    "35-44 years old": 39,
    "45-54 years old": 49,
    "55-64 years old": 59,
    "65 years or older": 70
}
insert_mapped_column(df, "Age", "AgeNum", age_map)

#Age zu numerischen Werten mappen, immer größter Wert der Spalte
age_map2 = {
    "Under 18 years old": 18,
    "18-24 years old": 24,
    "25-34 years old": 34,
    "35-44 years old": 44,
    "45-54 years old": 54,
    "55-64 years old": 64,
    "65 years or older": 100
}
insert_mapped_column(df, "Age", "MaxAge", age_map2)


#Über 65-jährige entfernen
df = df[df['AgeNum'] <= 65]

multi_select_cols = [
    'LanguageHaveWorkedWith', 'LanguageWantToWorkWith', 'DatabaseHaveWorkedWith', 'DatabaseWantToWorkWith', 'PlatformHaveWorkedWith', 'PlatformWantToWorkWith', 'WebframeHaveWorkedWith', 'WebframeWantToWorkWith', 'DevEnvsHaveWorkedWith', 'DevEnvsWantToWorkWith', 'OfficeStackAsyncHaveWorkedWith', 'OfficeStackAsyncWantToWorkWith', 'AIModelsHaveWorkedWith',     'AIModelsWantToWorkWith', 'AIAgent_Uses'
]

exclude_columns = ["Country", "Currency"]
category_columns = df.select_dtypes(include=["object"]).columns.tolist()

for col in category_columns:
    if col not in exclude_columns:
        df[col] = df[col].apply(to_lowercase)

for col in multi_select_cols:
    df[col] = df[col].apply(clean_multi_select)

# Keep only rows where WorkExp is NOT greater than (MaxAge - 16)
df = df[~(df['WorkExp'] > (df['MaxAge'] - 16))]

# Keep only rows where YearsCode is NOT greater than (MaxAge - 16)
df = df[~(df['YearsCode'] > (df['MaxAge'] - 6))]


# Mit Currency Converter Jahresgehalt in USD umwandeln

# Currency Spalte alles nach den ersten 3 Buchstaben abschneiden
df['Currency'] = df['Currency'].str[:3]

# Spalte CompTotal in USD umwandeln und in Spalte convertedCompTotal speichern

def convert_to_usd(currency, comp):
    if currency not in c.currencies:
        return np.nan
    if currency == "RUB":
        converted = c.convert(comp, currency, 'USD', date=date(2022, 3, 1))
    elif currency == "HRK":
        converted = c.convert(comp, currency, 'USD', date=date(2022, 12, 30))
    else:
        converted = c.convert(comp, currency, 'USD', date=date(2025, 10, 6))
    return converted

df['ConvertedCompTotal'] = df.apply(lambda row: convert_to_usd(row['Currency'], row['CompTotal']), axis=1)




# 95. Perzentil bestimmen
p95 = df["ConvertedCompYearly"].quantile(0.95)

# Daten filtern
df = df[df["ConvertedCompYearly"] <= p95]

df.to_csv("survey_results_cleaned.csv", index=False)


In [ ]:
df

# 🧹 Datenbereinigung & Vereinheitlichung – To-Do Liste

Diese To-Do-Liste beschreibt alle notwendigen Schritte, um die Survey-CSV-Datei zu bereinigen, zu vereinheitlichen und für spätere Analysen oder Visualisierungen nutzbar zu machen.

---

## 1. Fehlende Werte standardisieren

### Kategoriale Spalten
- `NaN` → **"Keine Angabe"**
- Datentyp `object` beibehalten

### Numerische Spalten
- `NaN` **nicht ersetzen**
- Datentyp `int`/`float` belassen
  → wichtig für statistische Auswertungen (Durchschnitt, Median, Histogramme)

In [ ]:
# Alle Spalten mit numerischen Werten in einen DataFrame packen
numeric_columns = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Alle Spalten mit nicht numerischen Werten in einen DataFrame packen
category_columns = df.select_dtypes(include=["object"]).columns.tolist()

print("Anzahl numerischer Spalten:", len(numeric_columns)) #Anzahl: 42
print("Anzahl kategorischer Spalten:", len(category_columns)) #Anzahl: 106

print("\nBeispiele numerischer Spalten:", numeric_columns[:10])
print("\nBeispiele kategorischer Spalten:", category_columns[:10])

## 2. Kategorische Text-Spalten bereinigen

### Ziel
Eine einheitliche und konsistente Darstellung, um spätere Gruppierungen und Analysen zu erleichtern.

### Maßnahmen
- Whitespace entfernen (Trimmen)
- Einheitliche Groß-/Kleinschreibung (z. B. `title()` oder `lower()`)
- Zusammenführen identischer kategorischer Werte mit unterschiedlicher Schreibweise
  *Beispiel: „Self taught“ und „self-taught“*
- Optional: Seltene Kategorien in **"Other"** gruppieren

### Beispiele betroffener Spalten
- `MainBranch`
- `EdLevel`
- `Employment`
- `Country`
- `OrgSize`
- `Industry`
- `AISelect`
- `AIPrimaryUse`

In [ ]:
# Currency außen vor wegen den Währungscodes wie USD oder EUR
exclude_columns = ["Country", "Currency"]

for col in category_columns:
    if col not in exclude_columns:
        df[col] = df[col].apply(to_lowercase)

## 3. Mehrfachauswahl-Spalten vereinheitlichen (`;`-getrennte Werte)

### Typische Probleme
- Uneinheitliche Formatierungen
- Semikolon-separierte Werte
- NaN-Werte
- Inkonsistente Reihenfolgen

### Maßnahmen
- Aufsplitten in Listen
  `"Python; JavaScript"` → `["Python", "JavaScript"]`
- Werte trimmen
- Duplikate in Listen entfernen
- Optionale alphabetische Sortierung der Werte
- NaN → **leere Liste** oder **"Keine Angabe"**

### Beispiele
- `DevType`
- `LanguageHaveWorkedWith`
- `LanguageWantToWorkWith`
- `DatabaseHaveWorkedWith`
- `ToolsTechHaveWorkedWith`
- `PlatformHaveWorkedWith`

In [ ]:
# Ausgabe aller Spalten, die Semikolon-getrennt sind
[col for col in df.columns if df[col].astype(str).str.contains(";").any()]

In [ ]:
# Alle Semikolon-getrennten Spalten in einer Liste
multi_select_cols = [
 'EmploymentAddl','LearnCode','AILearnHow','TechEndorse_13_TEXT','TechOppose_15_TEXT',
 'JobSatPoints_15_TEXT','LanguageHaveWorkedWith','LanguageWantToWorkWith','LanguageAdmired',
 'LanguagesHaveEntry','LanguagesWantEntry','DatabaseHaveWorkedWith','DatabaseWantToWorkWith',
 'DatabaseAdmired','DatabaseHaveEntry','DatabaseWantEntry','PlatformHaveWorkedWith',
 'PlatformWantToWorkWith','PlatformAdmired','PlatformWantEntry','WebframeHaveWorkedWith',
 'WebframeWantToWorkWith','WebframeAdmired','WebframeHaveEntry','WebframeWantEntry',
 'DevEnvsHaveWorkedWith','DevEnvsWantToWorkWith','DevEnvsAdmired','DevEnvHaveEntry',
 'DevEnvWantEntry','OpSysPersonal use','OpSysProfessional use',
 'OfficeStackAsyncHaveWorkedWith','OfficeStackAsyncWantToWorkWith','OfficeStackAsyncAdmired',
 'OfficeStackHaveEntry','CommPlatformHaveWorkedWith','CommPlatformWantToWorkWith',
 'CommPlatformAdmired','CommPlatformHaveEntr','CommPlatformWantEntr',
 'AIModelsHaveWorkedWith','AIModelsWantToWorkWith','AIModelsAdmired',
 'AIToolCurrently partially AI',"AIToolDon't plan to use AI for this task",
 'AIToolPlan to partially use AI','AIToolPlan to mostly use AI','AIToolCurrently mostly AI',
 'AIFrustration','AIExplain','AIAgent_Uses','AgentUsesGeneral',
 'AIAgentImpactSomewhat agree','AIAgentImpactNeutral','AIAgentImpactSomewhat disagree',
 'AIAgentImpactStrongly agree','AIAgentImpactStrongly disagree',
 'AIAgentChallengesNeutral','AIAgentChallengesSomewhat disagree',
 'AIAgentChallengesStrongly agree','AIAgentChallengesSomewhat agree',
 'AIAgentChallengesStrongly disagree','AIAgentKnowledge','AIAgentKnowWrite',
 'AIAgentOrchestration','AIAgentOrchWrite','AIAgentObserveSecure','AIAgentObsWrite',
 'AIAgentExternal','AIAgentExtWrite','AIHuman','AIOpen'
]

In [ ]:
def clean_multi_select(value):
    if pd.isna(value):
        return []

    splitted = str(value).split(";")

    cleaned = []
    for p in splitted:
        clean_text(p)
        if p:
            cleaned.append(p)

    cleaned = list(set(cleaned))

    cleaned.sort()

    return cleaned

In [ ]:
for col in multi_select_cols:
    df[col] = df[col].apply(clean_multi_select)

In [ ]:
df['YearsCode']

In [ ]:
df.to_csv("survey_results_shortened.csv", index=False)


## Filter die Zeilen raus, die mehr Berufs-Erfahrung haben als Lebensalter - 16 Jahre

In [ ]:
# Keep only rows where WorkExp is NOT greater than (MaxAge - 16)
df = df[~(df['WorkExp'] > (df['MaxAge'] - 16))]

# Display the first few rows of the filtered dataframe
df.head()

## Filter die Zeilen raus, die mehr Coding-Erfahrung haben als Lebensalter - 6 Jahre

In [ ]:
# Keep only rows where WorkExp is NOT greater than (MaxAge - 16)
df = df[~(df['YearsCode'] > (df['MaxAge'] - 6))]

# Display the first few rows of the filtered dataframe
df.head()

In [ ]:
# Mit Currency Converter Jahresgehalt in USD umwandeln

# Currency Spalte alles nach den ersten 3 Buchstaben abschneiden
df['Currency'] = df['Currency'].str[:3]

# Spalte CompTotal in USD umwandeln und in Spalte convertedCompTotal speichern

def convert_to_usd(currency, comp):
    if currency not in c.currencies:
        return np.nan
    if currency == "RUB":
        converted = c.convert(comp, currency, 'USD', date=date(2022, 3, 1))
    elif currency == "HRK":
        converted = c.convert(comp, currency, 'USD', date=date(2022, 12, 30))
    else:
        converted = c.convert(comp, currency, 'USD', date=date(2025, 10, 6))
    return converted

df['ConvertedCompTotal'] = df.apply(lambda row: convert_to_usd(row['Currency'], row['CompTotal']), axis=1)




In [ ]:
df.to_csv("survey_results_test.csv", index=False)